In [155]:
import pandas as pd
import numpy as np
from function_file import standardize_columns, normalize_data
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingRegressor, BaggingClassifier,RandomForestRegressor,AdaBoostRegressor, AdaBoostClassifier, GradientBoostingClassifier


from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [156]:
df = pd.read_csv("survey.csv")
print(df.head(5))

             Timestamp  Age  Gender         Country state self_employed  \
0  2014-08-27 11:29:31   37  Female   United States    IL           NaN   
1  2014-08-27 11:29:37   44       M   United States    IN           NaN   
2  2014-08-27 11:29:44   32    Male          Canada   NaN           NaN   
3  2014-08-27 11:29:46   31    Male  United Kingdom   NaN           NaN   
4  2014-08-27 11:30:22   31    Male   United States    TX           NaN   

  family_history treatment work_interfere    no_employees  ...  \
0             No       Yes          Often            6-25  ...   
1             No        No         Rarely  More than 1000  ...   
2             No        No         Rarely            6-25  ...   
3            Yes       Yes          Often          26-100  ...   
4             No        No          Never         100-500  ...   

                leave mental_health_consequence phys_health_consequence  \
0       Somewhat easy                        No                      No   
1 

In [157]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Timestamp                  1259 non-null   object
 1   Age                        1259 non-null   int64 
 2   Gender                     1259 non-null   object
 3   Country                    1259 non-null   object
 4   state                      744 non-null    object
 5   self_employed              1241 non-null   object
 6   family_history             1259 non-null   object
 7   treatment                  1259 non-null   object
 8   work_interfere             995 non-null    object
 9   no_employees               1259 non-null   object
 10  remote_work                1259 non-null   object
 11  tech_company               1259 non-null   object
 12  benefits                   1259 non-null   object
 13  care_options               1259 non-null   object
 14  wellness

In [158]:
df.shape

(1259, 27)

In [159]:
df.columns

Index(['Timestamp', 'Age', 'Gender', 'Country', 'state', 'self_employed',
       'family_history', 'treatment', 'work_interfere', 'no_employees',
       'remote_work', 'tech_company', 'benefits', 'care_options',
       'wellness_program', 'seek_help', 'anonymity', 'leave',
       'mental_health_consequence', 'phys_health_consequence', 'coworkers',
       'supervisor', 'mental_health_interview', 'phys_health_interview',
       'mental_vs_physical', 'obs_consequence', 'comments'],
      dtype='object')

- For this project I would b econsidering Treatment as Target variable
- There are some columns which are related to this column directl, so i will drop these columns
- also as many columns present to be considered as feature but i for project to keep simple i am 

In [160]:
# Updated for Tech Survey Dataset
selected_columns = [
    # Your latest requested columns (Stigma & Culture)
    'phys_health_consequence',
    'coworkers',
    
    # HIGH IMPACT columns (Essential for getting above 0.33 accuracy)
    'family_history',
    'benefits',
    'Age',
    'work_interfere',
    
    # Target column (In this dataset, 'treatment' is the goal)
    'treatment'  
]

# Apply the selection to your new dataframe
df = df[selected_columns]

# View remaining columns
print("Columns successfully updated for Tech Survey Analysis:")
print(df.columns)

Columns successfully updated for Tech Survey Analysis:
Index(['phys_health_consequence', 'coworkers', 'family_history', 'benefits',
       'Age', 'work_interfere', 'treatment'],
      dtype='object')


In [161]:
# Datatypes of columns
df.dtypes

phys_health_consequence    object
coworkers                  object
family_history             object
benefits                   object
Age                         int64
work_interfere             object
treatment                  object
dtype: object

Checking for missing values

In [162]:
((df.isna().sum())/len(df))*100

phys_health_consequence     0.000000
coworkers                   0.000000
family_history              0.000000
benefits                    0.000000
Age                         0.000000
work_interfere             20.969023
treatment                   0.000000
dtype: float64

**Perform train test**

In [163]:
features = df.drop(columns=['treatment'])
target = df["treatment"]

In [164]:
df["treatment"]

0       Yes
1        No
2        No
3       Yes
4        No
       ... 
1254    Yes
1255    Yes
1256    Yes
1257     No
1258    Yes
Name: treatment, Length: 1259, dtype: object

In [165]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=17)

In [166]:
# 1. Standardize and Map the Target (treatment)
y_train = y_train.map({'Yes': 1, 'No': 0})
y_test = y_test.map({'Yes': 1, 'No': 0})

In [167]:
print(X_train.shape)
print(X_test.shape)

(1007, 6)
(252, 6)


In [168]:
X_train

,phys_health_consequence,coworkers,family_history,benefits,Age,work_interfere
132,No,Yes,No,Don't know,27,Never
909,Maybe,Yes,No,Yes,48,Never
744,Maybe,Some of them,No,Yes,36,Sometimes
1012,Maybe,Some of them,Yes,Yes,24,Sometimes
1257,No,No,No,No,46,NaN
...,...,...,...,...,...,...
278,No,Some of them,No,Yes,28,Rarely
752,No,Some of them,Yes,Don't know,37,Sometimes
406,Maybe,Yes,Yes,Yes,33,Never
143,No,Some of them,No,Yes,-29,NaN


In [169]:
X_train_encoded = standardize_columns(X_train)

Doing normalization so that all features are in one range

In [170]:
normalizer = MinMaxScaler()

In [171]:
X_train_encoded

,phys_health_consequence,coworkers,family_history,benefits,Age,work_interfere
132,0,2,0,0.5,27,Never
909,1,2,0,1.0,48,Never
744,1,1,0,1.0,36,Sometimes
1012,1,1,1,1.0,24,Sometimes
1257,0,0,0,0.0,46,NaN
...,...,...,...,...,...,...
278,0,1,0,1.0,28,Rarely
752,0,1,1,0.5,37,Sometimes
406,1,2,1,1.0,33,Never
143,0,1,0,1.0,-29,NaN


In [172]:
normalizer.fit(X_train_encoded)

ValueError: could not convert string to float: 'Never'

In [ ]:
X_train_norm = normalize_data(X_train_encoded, normalizer)


In [ ]:
X_train_norm

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors = 10)

In [ ]:
X_train_norm.shape

In [ ]:
model.fit(X_train_norm,y_train)

In [ ]:
#X_test = one_hot_encoding(X_test)
X_test_stand= standardize_columns(X_test)
X_test_norm = normalize_data(X_test_stand, normalizer)

In [ ]:
model.score(X_test_norm, y_test)

Trying liniar regression

In [ ]:
lin_reg = LinearRegression()

In [ ]:
lin_reg.fit(X_train_norm, y_train)

In [ ]:
lin_reg.score(X_test_norm, y_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and fit
df = RandomForestClassifier(n_estimators=100, random_state=17)
df.fit(X_train_norm, y_train)

# Score
print(df.score(X_test_norm, y_test))

In [ ]:
tree = DecisionTreeClassifier(max_depth=10)

In [ ]:
tree.fit(X_train_norm, y_train)

In [ ]:
tree.score(X_test_norm,y_test)

In [ ]:
df_imp = pd.DataFrame({
    "features": X_test_norm.columns,
    'impoertance': tree.feature_importances_
})

df_imp

Bagging and Pasting

In [ ]:
bagging_reg = BaggingClassifier(DecisionTreeClassifier(max_depth=20),
                               n_estimators=100,
                               max_samples = 1000)

In [ ]:
bagging_reg.fit(X_train_norm, y_train)

In [ ]:
forest = RandomForestClassifier(n_estimators=100,
                             max_depth=20)

In [ ]:
pred = bagging_reg.predict(X_test_norm)
print("R2 score", bagging_reg.score(X_test_norm, y_test))

Random Patches

In [ ]:
forest = RandomForestClassifier(n_estimators=100,
                             max_depth=20)
forest.fit(X_train_norm, y_train)

In [ ]:
pred = forest.predict(X_test_norm)
print("R2 score", forest.score(X_test_norm, y_test))

Adaboost

In [ ]:
ada_reg = AdaBoostClassifier(DecisionTreeClassifier(max_depth=20),
                            n_estimators=100)
ada_reg.fit(X_train_norm, y_train)

In [ ]:
pred = ada_reg.predict(X_test_norm)

print("R2 score", ada_reg.score(X_test_norm, y_test))

Gradient Boosting

In [ ]:
gb_reg = GradientBoostingClassifier(max_depth=20,
                                   n_estimators=100)
gb_reg.fit(X_train_norm, y_train)

In [ ]:
pred = gb_reg.predict(X_test_norm)
print("R2 score", gb_reg.score(X_test_norm, y_test))

In [ ]:
import joblib

# 'model' is your trained DecisionTreeClassifier
joblib.dump(model, 'tech_survey_model.pkl')
print("Model saved successfully!")

In [ ]:
# 'normalizer' is the name of the MinMaxScaler you fitted in your notebook
joblib.dump(normalizer, 'scaler.pkl')

print("Success! 'scaler.pkl' has been created in your project folder.")

In [ ]:
# This prints the EXACT order the model learned
print("Order for Streamlit:", X_train.columns.tolist())